# Task 3: RAG Pipeline Construction and Evaluation

This notebook implements the retrieval and generation pipeline using the
pre-built FAISS vector store and evaluates its effectiveness using
qualitative analysis.

## Load Vector Store

In [ ]:
import faiss
import pickle
from utils.paths import FAISS_INDEX_PATH, METADATA_PATH

index = faiss.read_index(str(FAISS_INDEX_PATH))

with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

print("FAISS index size:", index.ntotal)
print("Metadata entries:", len(metadata))

## Initialize Pipeline


In [ ]:

from src.retriever import ComplaintRetriever
from src.generator import AnswerGenerator
from src.rag_pipeline import RAGPipeline

retriever = ComplaintRetriever(index=index, metadata=metadata)
generator = AnswerGenerator()
rag_pipeline = RAGPipeline(retriever, generator)

## Qualitative Evaluation Questions

In [ ]:
evaluation_questions = [
    "What are the most common credit card billing issues reported by customers?",
    "What complaints do customers raise about savings account fees?",
    "Are there recurring problems with money transfer delays?",
    "What issues do personal loan customers face during repayment?",
    "Do customers report problems with credit card dispute resolution?"
]

### Run Evaluation


In [ ]:

results = []

for q in evaluation_questions:
    output = rag_pipeline.run(q, k=5)
    results.append(output)
    print(f"Question: {q}")
    print(f"Answer: {output['answer']}")  
    print(f"Sources: {output['sources']}\n")
    print("-" * 50)




### Create Evaluation Table


In [ ]:

from src.evaluation import evaluate_responses

evaluation_df = evaluate_responses(results)
evaluation_df


## Evaluation Analysis

- Most responses are grounded in retrieved complaint narratives.
- The system performs well for high-volume products such as credit cards.
- Some answers could be improved through better summarization and re-ranking.
- Future improvements include hybrid retrieval and structured outputs.